# BTP: explainable AI-content detection, Kaggle GPU runner
Settings: **Accelerator = GPU T4**, **Internet = On**. Then *Save Version -> Save & Run All (Commit)* so it runs in the background. Outputs (models, features, metrics, logs) appear under *Output*.

In [ ]:
# Stop early with a clear message if the session is misconfigured
import shutil, socket
problems = []
if not shutil.which('nvidia-smi'):
    problems.append('No GPU: right panel -> Session options -> Accelerator -> GPU T4 x2 (needs phone verification)')
try:
    socket.create_connection(('github.com', 443), timeout=10).close()
except OSError:
    problems.append('No internet: right panel -> Session options -> Internet -> On (needs phone verification)')
if problems:
    raise RuntimeError('FIX SESSION SETTINGS FIRST:\n- ' + '\n- '.join(problems))
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /kaggle/working
!rm -rf BTP-1 && git clone -q --depth 1 https://github.com/lakshayg2005/BTP-1.git
%cd BTP-1
# big TRAINING images go to /kaggle/tmp (not saved); test sets, features and models stay in the saved output
!mkdir -p /kaggle/tmp/hf /kaggle/tmp/images_train /kaggle/tmp/sid_train data/images data/sid
!ln -sfn /kaggle/tmp/images_train data/images/train && ln -sfn /kaggle/tmp/sid_train data/sid/train
import os


In [ ]:
# RESUME: if an earlier version's output is attached (Add Input -> Your Work -> this notebook), reuse its finished work
import glob, shutil
for prev in glob.glob('/kaggle/input/**/BTP-1', recursive=True):
    print('resuming from', prev)
    for sub in ['features', 'models', 'logs', 'data/images/test', 'data/sid/test', 'data/audio']:
        if os.path.exists(f'{prev}/{sub}'):
            shutil.copytree(f'{prev}/{sub}', sub, dirs_exist_ok=True)
print('features:', sorted(os.listdir('features')) if os.path.exists('features') else 'none yet')
print('models:', sorted(os.listdir('models')) if os.path.exists('models') else 'none yet')
os.environ.update(PY='python', HF_HOME='/kaggle/tmp/hf', TMPDIR_BTP='/kaggle/tmp/t',
                  IMG_TRAIN='3000', IMG_TEST='500', SID_TRAIN='1500', SID_TEST='500', TXT_TRAIN='2000', TXT_TEST='400', BTP_TEXT_LM_SIZE='1.5B')

In [ ]:
!bash scripts/run_phase.sh deps 2>&1 | tail -3
!python -c "import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())"

## Phase 1: image AI detector

In [ ]:
!bash scripts/run_phase.sh image 2>&1 | grep --line-buffered -v -E 'UNEXPECTED|^Notes:|can be ignored|^[[:space:]]*$'

## Phase 2: edit localizer (SID-Set masks + synthetic edits)

In [ ]:
!bash scripts/run_phase.sh tamper 2>&1 | grep --line-buffered -v -E 'UNEXPECTED|^Notes:|can be ignored|^[[:space:]]*$'

## Phase 3: audio

In [ ]:
!bash scripts/run_phase.sh audio 2>&1 | grep --line-buffered -v -E 'UNEXPECTED|^Notes:|can be ignored|^[[:space:]]*$'

## Phase 4: text

In [ ]:
!bash scripts/run_phase.sh text 2>&1 | grep --line-buffered -v -E 'UNEXPECTED|^Notes:|can be ignored|^[[:space:]]*$'

## Evaluation: results tables + example explanations

In [ ]:
!bash scripts/run_phase.sh evaluate 2>&1 | grep --line-buffered -v -E 'UNEXPECTED|^Notes:|can be ignored|^[[:space:]]*$'

In [ ]:
from IPython.display import Markdown, display
if os.path.exists('results/summary.md'):
    display(Markdown(open('results/summary.md').read()))
else:
    print('No summary yet: check the phase logs above for [warn] or Traceback lines')

In [ ]:
# keep small outputs only; big data stays in /kaggle/tmp
!cd /kaggle/working && zip -qr btp_outputs.zip BTP-1/models BTP-1/results BTP-1/logs BTP-1/features -x '*.npz' && ls -la btp_outputs.zip

## Final status
Kaggle marks a run *successful* whenever the cells finish, even if a script inside failed. This checklist is the real verdict: the run is marked failed if anything is missing (outputs are already saved).

In [ ]:
expected = {'image AI detector': 'models/image_head.joblib', 'edit localizer': 'models/tamper_head.pt',
            'audio detector': 'models/audio_head.joblib', 'text detector': 'models/text_head.joblib',
            'evaluation report': 'results/summary.md'}
missing = [name for name, f in expected.items() if not os.path.exists(f)]
for name, f in expected.items():
    print(('OK      ' if os.path.exists(f) else 'MISSING ') + f'{name:20s} {f}')
if missing:
    raise RuntimeError(f'Run incomplete, missing: {missing}. Attach this version as input and run again.')
print('ALL DONE')

## If a run stops early
Its output is still saved. Start a new version: *Add Input -> Your Work* -> pick this notebook (the stopped version), then *Save Version -> Save & Run All* again. Finished steps are skipped.

## Live demo (interactive sessions only)
Open the notebook in edit mode, run all cells, then this one prints a public gradio link.

In [ ]:
import os
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') == 'Interactive':
    !python -m app.demo --share
else:
    print('Batch run: demo skipped')